# Live PSU plot with Plotly

Sweep a simulated power supply and watch `PlotlyPublisher` plot each reading live in this notebook.

**Before you start**

1. Start the simulated PSU in a terminal and leave it running:

   ```bash
   uv run python -m instro.psu.scpi_sim_server
   ```

2. Open this notebook with the `plot` extra and Jupyter. Jupyter needs `anywidget` alongside it to draw the live figure:

   ```bash
   uv run --extra plot --with jupyter --with anywidget jupyter lab examples/publishers/publish_plotly_notebook.ipynb
   ```

In [1]:
import time

from instro.lib.publishers import PlotlyPublisher
from instro.psu import InstroPSU
from instro.psu.drivers import SimulatedPSU

VISA_RESOURCE = "TCPIP0::127.0.0.1::5025::SOCKET"

Called with no figure, `PlotlyPublisher` creates a live `go.FigureWidget` and displays it below this cell. To style the plot, pass your own `go.FigureWidget` and display it instead.

In [2]:
plot_publisher = PlotlyPublisher()

psu = InstroPSU(
    name="myPSU",
    driver=SimulatedPSU(VISA_RESOURCE),
    num_channels=2,
    publishers=[plot_publisher],
)
psu.open()

FigureWidget({
    'data': [], 'layout': {'template': '...', 'xaxis': {'type': 'date'}}
})

Run the sweep and watch the figure above update as each reading arrives.

In [3]:
psu.set_current_limit(0.2, channel=1)
psu.output_enable(True, channel=1)

for v in range(10):
    psu.set_voltage(v, channel=1)
    time.sleep(0.5)
    psu.get_voltage(channel=1)
    psu.get_current(channel=1)

RuntimeError: Simulated PSU reported error: -222,"Data out of range"

Turn the output off and close the PSU when you're done. Closing also writes any remaining points to the figure.

In [19]:
psu.output_enable(False, channel=1)
psu.close()